# 0️⃣ Расширения GHC: бриф

**Language extensions** — это управляемые расширения стандарта Haskell, которые включаются прагмой `{-# LANGUAGE ... #-}` или командой `:set -X...` в REPL/IHaskell.

Стандарт Haskell 2010 намеренно консервативен; расширения дают доступ к более выразительным возможностям системы типов и синтаксиса, которые либо ещё не вошли в стандарт, либо принципиально выходят за его рамки.

## О чём этот ноутбук

Каждый ноутбук курса включает только те расширения, которые действительно нужны для его кода.
Здесь — краткий справочник: **что, зачем, в чём соль** для каждого из 26 расширений курса.

## Правило курса

Расширения включаются в **setup-ячейке** каждого ноутбука через `:set -X...`, например:

```
:set -XOverloadedStrings -XLambdaCase -XScopedTypeVariables
```

In [1]:
:set -XOverloadedStrings
:set -XLambdaCase
:set -XTupleSections
:set -XScopedTypeVariables
:set -XRankNTypes
:set -XExistentialQuantification
:set -XFlexibleInstances
:set -XTypeSynonymInstances
:set -XFlexibleContexts
:set -XMultiParamTypeClasses
:set -XFunctionalDependencies
:set -XInstanceSigs
:set -XDefaultSignatures
:set -XDeriveFunctor
:set -XDeriveFoldable
:set -XDeriveTraversable
:set -XDeriveGeneric
:set -XGeneralizedNewtypeDeriving
:set -XTypeOperators
:set -XTypeFamilies
:set -XGADTs
:set -XDataKinds
:set -XPolyKinds
putStrLn "'05 SETUP OK"

'05 SETUP OK

## Содержание

1. [Синтаксис и литералы](#overloadedstrings): OverloadedStrings, LambdaCase, TupleSections
2. [Forall-полиморфизм](#scopedtypevariables): ScopedTypeVariables, RankNTypes, ExistentialQuantification
3. [Классы типов](#flexibleinstances): FlexibleInstances/TypeSynonymInstances, FlexibleContexts, MultiParamTypeClasses/FunctionalDependencies, InstanceSigs/DefaultSignatures
4. [Deriving](#deriving): DeriveFunctor/Foldable/Traversable/Generic, GeneralizedNewtypeDeriving
5. [Уровень типов](#typeoperators): TypeOperators, TypeFamilies, GADTs, DataKinds/PolyKinds
6. [Стрелки (Arrows)](#arrows): proc-нотация
7. [Метапрограммирование](#templatehaskell): TemplateHaskell, QuasiQuotes
8. [За пределами курса](#beyond)
9. [Итоговая таблица](#table)

<a id="overloadedstrings"></a>
## Синтаксис и литералы

### OverloadedStrings
**В чём соль:** строковый литерал `"hello"` перегружается — его тип становится `IsString a => a`, а не фиксированным `String`. Позволяет писать `"key"` там, где ожидается `Text`, `ByteString` и т.д., без явных конверсий.
Ноутбуки-потребители: SubjectiveModeling, Toposes, Uncertainty

<a id="lambdacase"></a>
### LambdaCase
**В чём соль:** синтаксический сахар `\case` эквивалентен `\x -> case x of`. Убирает лишнее имя-пустышку и делает point-free стиль доступным там, где нужен pattern-matching.
Ноутбуки-потребители: Adjunctions, Uncertainty

<a id="tuplesections"></a>
### TupleSections
**В чём соль:** `(,True)` — частично применённый конструктор пары; `(,True) 5` даёт `(5,True)`. Удобно в комбинации с `map`, `fmap` и стрелками.
Ноутбуки-потребители: Adjunctions, Arrows, FoldableTraversable, Optics

In [2]:
import Data.String (IsString(..))

-- OverloadedStrings: кастомный тип, реализующий IsString
newtype MyStr = MyStr String deriving (Show)

instance IsString MyStr where
  fromString s = MyStr s

greet :: MyStr
greet = "Привет, мир"

-- LambdaCase: сопоставление без имени переменной
describeInt :: Maybe Int -> Int
describeInt = (\case { Just x -> x; Nothing -> 0 })

-- TupleSections: частичное применение конструктора пары
pairs :: [(Int, Bool)]
pairs = map (,True) [1, 2, 3]

putStrLn ("OverloadedStrings: " ++ show greet)
putStrLn ("LambdaCase: " ++ show (describeInt (Just 42)) ++ ", " ++ show (describeInt Nothing))
putStrLn ("TupleSections: " ++ show pairs)

Line 7: Eta reduce
Found:
fromString s = MyStr s
Why not:
fromString = MyStrLine 14: Redundant bracket
Found:
(\case
   Just x -> x
   Nothing -> 0)
Why not:
\case
  Just x -> x
  Nothing -> 0

OverloadedStrings: MyStr "\1055\1088\1080\1074\1077\1090, \1084\1080\1088"

LambdaCase: 42, 0

TupleSections: [(1,True),(2,True),(3,True)]

<a id="scopedtypevariables"></a>
## Forall-полиморфизм

### ScopedTypeVariables
**В чём соль:** переменные типа из сигнатуры функции (введённые явным `forall`) становятся видны в теле, в том числе в `where`-клаузулах и вложенных аннотациях. Без этого расширения вложенные аннотации типов создают независимые переменные.
Ноутбуки-потребители: Adjunctions, AlgebrasCoalgebras, BaseHaskell, ComonadTransformers, DistributedHaskell, FoldableTraversable, FunctorHierarchy, GPUHaskell, KanExtensions, MetaProgramming, Monads, MonadTransformers, Optics, Profunctors, SubjectiveModeling, Toposes, TypeAlgebra, Uncertainty, YonedaLemma

<a id="rankntypes"></a>
### RankNTypes
**В чём соль:** аргумент функции сам может быть полиморфным — т.е. иметь тип `forall a. ...`. Стандарт допускает только ранг-1 (polym. только на верхнем уровне). Нужен для `runST`, линз, CPS и других паттернов.
Ноутбуки-потребители: Adjunctions, AlgebrasCoalgebras, BaseHaskell, ComonadTransformers, DistributedHaskell, FunctorHierarchy, GPUHaskell, KanExtensions, Monads, MonadTransformers, Optics, Profunctors, SubjectiveModeling, Toposes, TypeAlgebra, Uncertainty, YonedaLemma

<a id="existentialquantification"></a>
### ExistentialQuantification
**В чём соль:** тип-параметр конструктора данных скрывается снаружи — `forall a. Show a => Box a` позволяет класть в один список значения разных типов, объединённых только ограничением.
Ноутбуки-потребители: KanExtensions, Profunctors, YonedaLemma

In [3]:
-- ScopedTypeVariables: явный forall, чтобы использовать 'a' в where
reverseGeneric :: forall a. [a] -> [a]
reverseGeneric xs = go xs []
  where
    go :: [a] -> [a] -> [a]
    go []     acc = acc
    go (h:t)  acc = go t (h:acc)

-- RankNTypes: аргумент сам полиморфен
applyBoth :: (forall a. a -> a) -> (Int, Bool) -> (Int, Bool)
applyBoth f (x, y) = (f x, f y)

-- ExistentialQuantification: гетерогенный контейнер
data Showable = forall a. Show a => MkShowable a

showIt :: Showable -> String
showIt (MkShowable x) = show x

things :: [Showable]
things = [MkShowable (42 :: Int), MkShowable True, MkShowable "hello"]

putStrLn ("ScopedTypeVariables: " ++ show (reverseGeneric [1,2,3::Int]))
putStrLn ("RankNTypes: " ++ show (applyBoth id (7, False)))
putStrLn ("ExistentialQuantification: " ++ unwords (map showIt things))

ScopedTypeVariables: [3,2,1]

RankNTypes: (7,False)

ExistentialQuantification: 42 True "hello"

<a id="flexibleinstances"></a>
## Классы типов

### FlexibleInstances + TypeSynonymInstances
**В чём соль:** Haskell 2010 разрешает инстансы только вида `C T` или `C (T a b)` с переменными-аргументами. FlexibleInstances снимает это ограничение — можно писать `instance C [Int]` или `instance C (Maybe String)`. TypeSynonymInstances разрешает использовать синонимы типов (`String`) как голову инстанса.
Ноутбуки-потребители: Adjunctions, Arrows, ComonadTransformers, FunctorHierarchy, KanExtensions, MetaProgramming, Optics, Profunctors, SubjectiveModeling, Toposes, Uncertainty, YonedaLemma

<a id="flexiblecontexts"></a>
### FlexibleContexts
**В чём соль:** контекст ограничений может содержать произвольные применения классов, не только `C a` с голой переменной. Например, `(Show (f a)) =>` становится легальным.
Ноутбуки-потребители: Adjunctions, Arrows, ComonadTransformers, FunctorHierarchy, KanExtensions, MetaProgramming, Optics, Profunctors, SubjectiveModeling, Toposes, Uncertainty, YonedaLemma

<a id="multiparamtypeclasses"></a>
### MultiParamTypeClasses + FunctionalDependencies
**В чём соль:** класс типов может принимать несколько параметров: `class Convertible a b`. FunctionalDependencies (`| a -> b`) задают функциональные зависимости между параметрами, помогая type-checker'у выводить типы однозначно.
Ноутбуки-потребители: Adjunctions, ComonadTransformers, FunctorHierarchy, Profunctors, Toposes

<a id="instancesigs"></a>
### InstanceSigs + DefaultSignatures
**В чём соль:** InstanceSigs позволяет явно писать сигнатуры методов внутри `instance`-блока — удобно для документирования и специализации. DefaultSignatures позволяют объявить реализацию метода по умолчанию с более конкретной сигнатурой (например, с дополнительным ограничением).
Ноутбуки-потребители: Adjunctions, Arrows, FoldableTraversable, FunctorHierarchy, Optics, Profunctors, Toposes

In [4]:
-- MultiParamTypeClasses + FunctionalDependencies
class Container c e | c -> e where
  empty  :: c
  insert :: e -> c -> c
  toList :: c -> [e]

instance Container [Int] Int where
  empty      = []
  insert x c = x : c
  toList     = id

-- FlexibleInstances: инстанс для конкретного типа [Char]
class Describable a where
  describe :: a -> String

instance Describable [Char] where
  describe s = "строка: " ++ s

instance Describable Int where
  describe n = "число: " ++ show n

-- InstanceSigs: явная сигнатура в инстансе
class Wrap a where
  wrap   :: a -> [a]
  wrap x = [x]           -- default

instance Wrap Int where
  wrap :: Int -> [Int]
  wrap x = [x, x]        -- переопределяем

let c0 = empty :: [Int]
    c1 = insert 3 (insert 1 (insert 2 c0))
putStrLn ("Container: " ++ show (toList c1))
putStrLn (describe ("мир" :: String))
putStrLn (describe (99 :: Int))
putStrLn ("InstanceSigs wrap: " ++ show (wrap (5 :: Int)))

Container: [3,1,2]

строка: мир

число: 99

InstanceSigs wrap: [5,5]

<a id="deriving"></a>
## Deriving

### DeriveFunctor / DeriveFoldable / DeriveTraversable
**В чём соль:** компилятор автоматически генерирует экземпляры `Functor`, `Foldable`, `Traversable` для пользовательских типов данных, следуя структуре конструкторов. Исключает boilerplate для деревьев, потоков, rose-trees и т.п.
Ноутбуки-потребители: Adjunctions, Comonads, ComonadTransformers, FoldableTraversable, Profunctors, SubjectiveModeling, Toposes, Uncertainty, YonedaLemma (Functor), FoldableTraversable (Foldable), FoldableTraversable (Traversable)

### DeriveGeneric
**В чём соль:** генерирует инстанс `GHC.Generics.Generic`, открывая доступ к структуре типа на уровне типов. Основа для библиотек `aeson`, `binary`, `lens` (deriving-through-Generic).
Ноутбуки-потребители: MetaProgramming

### GeneralizedNewtypeDeriving
**В чём соль:** `newtype` автоматически наследует инстансы обёрнутого типа — не только стандартные классы, но и любой класс, для которого реализация корректна через coerce. Устраняет необходимость писать делегирующие инстансы вручную.
Ноутбуки-потребители: SubjectiveModeling, Toposes, Uncertainty

In [5]:
-- DeriveFunctor, DeriveFoldable, DeriveTraversable
data Tree a = Leaf | Node (Tree a) a (Tree a)
  deriving (Functor, Foldable, Show)

-- GeneralizedNewtypeDeriving: Age наследует Num у Int
newtype Age = Age Int
  deriving (Show, Eq, Num)

let t = Node (Node Leaf 1 Leaf) 2 (Node Leaf 3 Leaf)
putStrLn ("fmap (*10) tree: " ++ show (fmap (*10) t))
putStrLn ("sum tree: " ++ show (sum t))

let a1 = Age 30
    a2 = Age 5
putStrLn ("Age: " ++ show a1 ++ " + " ++ show a2 ++ " = " ++ show (a1 + a2))

fmap (*10) tree: Node (Node Leaf 10 Leaf) 20 (Node Leaf 30 Leaf)

sum tree: 6

Age: Age 30 + Age 5 = Age 35

<a id="typeoperators"></a>
## Уровень типов

### TypeOperators
**В чём соль:** операторы (символьные идентификаторы) могут использоваться как конструкторы типов: `data f :+: g = ...`. Используется в GHC.Generics, optics-библиотеках.
Ноутбуки-потребители: Adjunctions, FunctorHierarchy, Optics, Profunctors

<a id="typefamilies"></a>
### TypeFamilies
**В чём соль:** функции на уровне типов — `type family Elem c where Elem [a] = a`. Позволяют выражать вычисления над типами в стиле функциональной зависимости, но более мощно и удобно, чем FunctionalDependencies.
Ноутбуки-потребители: Comonads, Toposes

<a id="gadts"></a>
### GADTs
**В чём соль:** каждый конструктор может специализировать возвращаемый тип — `IntE :: Int -> Expr Int`. Это делает невозможным «неправильное» построение значений и позволяет писать тотальные функции-интерпретаторы без `undefined`.
Ноутбуки-потребители: Toposes

<a id="datakinds"></a>
### DataKinds + PolyKinds
**В чём соль:** DataKinds «поднимает» типы данных на уровень сортов (kinds) — конструкторы `True`, `False` становятся типами сорта `Bool`. PolyKinds добавляет полиморфизм по сортам. Основа для type-level natural numbers, гетерогенных списков, phantom-типов.
Ноутбуки-потребители: Toposes

In [6]:
-- TypeFamilies: ассоциированное семейство типов
class Collection c where
  type Elem c
  fromList :: [Elem c] -> c
  size     :: c -> Int

instance Collection [a] where
  type Elem [a] = a
  fromList = id
  size     = length

-- GADTs: типобезопасное выражение
data Expr a where
  IntE  :: Int  -> Expr Int
  BoolE :: Bool -> Expr Bool
  If    :: Expr Bool -> Expr a -> Expr a -> Expr a

eval :: Expr a -> a
eval (IntE  n)    = n
eval (BoolE b)    = b
eval (If c t e)   = if eval c then eval t else eval e

-- TypeOperators: оператор-тип
data a :+: b = Inl a | Inr b deriving (Show)

let e1 = If (BoolE True) (IntE 42) (IntE 0)
    e2 = If (BoolE False) (IntE 1) (IntE 2)
putStrLn ("GADT eval: " ++ show (eval e1) ++ ", " ++ show (eval e2))
putStrLn ("TypeFamilies size: " ++ show (size (fromList [1,2,3] :: [Int])))
let v = Inl (42::Int) :: Int :+: Bool
putStrLn ("TypeOperators :+: " ++ show v)

GADT eval: 42, 2

TypeFamilies size: 3

TypeOperators :+: Inl 42

<a id="arrows"></a>
## Стрелки (Arrows)

### Arrows
**В чём соль:** расширение включает `proc`-нотацию — синтаксический сахар для комбинаторов класса `Arrow`. Стрелки обобщают монады: `proc x -> do ...` выглядит как do-нотация, но работает для любого `Arrow`, включая `Function`, `Kleisli`, парсер-комбинаторы.

```haskell
addA :: Arrow a => a (b, c) b -> a (b, c) c -> a (b, c) (b, c)
addA f g = proc x -> do
  y <- f -< x
  z <- g -< x
  returnA -< (y, z)
```

Ноутбуки-потребители: Arrows

> Полный разбор с примерами: **[Arrows.ipynb](Arrows.ipynb)**

<a id="templatehaskell"></a>
## Метапрограммирование

### TemplateHaskell + QuasiQuotes
**В чём соль:** TemplateHaskell — метапрограммирование времени компиляции: сплайсы `$(expr)` вставляют сгенерированный AST прямо в код. QuasiQuotes (`[myDSL| ... |]`) — встраиваемые DSL-литералы, разбираемые кастомным парсером.

Примеры применений: автогенерация `Lens`-полей (`makeLenses`), персистентные схемы Persistent, генерация FFI-биндингов, compile-time парсинг регулярных выражений.

Ноутбуки-потребители: MetaProgramming

> Полный разбор с примерами: **[MetaProgramming.ipynb](MetaProgramming.ipynb)**

<a id="beyond"></a>
## За пределами курса

Ниже — расширения, часто встречающиеся в реальных проектах, но не используемые в ноутбуках курса.

**TypeApplications** (`-XTypeApplications`): явное применение типов к полиморфным функциям — `read @Int "42"`. Убирает необходимость в вспомогательных аннотациях типов во многих местах.

**BangPatterns / StrictData** (`-XBangPatterns`, `-XStrictData`): управление строгостью на уровне паттернов (`!x`) или всего модуля. Используется для оптимизации пространственного потребления и устранения пространственных утечек.

**OverloadedRecordDot** (`-XOverloadedRecordDot`): синтаксис `record.field` для доступа к полям записей — как в большинстве других языков. Доступен начиная с GHC 9.2.

**PatternSynonyms** (`-XPatternSynonyms`): именованные алиасы для паттернов. Позволяют создавать стабильный публичный API поверх изменяемой внутренней структуры данных.

**LinearTypes** (`-XLinearTypes`): типы с линейными ограничениями — функция `a %1 -> b` обязана использовать аргумент ровно один раз. Открывает путь к safe manual memory management и resource tracking на уровне типов. Доступен с GHC 9.0.

<a id="table"></a>
## Итоговая таблица расширений курса

| Расширение | Зачем | Ноутбуки |
|---|---|---|
| `OverloadedStrings` | строковые литералы любого типа IsString | SubjectiveModeling, Toposes, Uncertainty |
| `LambdaCase` | \case вместо \x -> case x of | Adjunctions, Uncertainty |
| `TupleSections` | частичное применение конструктора кортежа: (,5) | Adjunctions, Arrows, FoldableTraversable, Optics |
| `ScopedTypeVariables` | переменные типа из сигнатуры видны в теле | Adjunctions, AlgebrasCoalgebras, BaseHaskell, ComonadTransformers, DistributedHaskell, FoldableTraversable, FunctorHierarchy, GPUHaskell, KanExtensions, MetaProgramming, Monads, MonadTransformers, Optics, Profunctors, SubjectiveModeling, Toposes, TypeAlgebra, Uncertainty, YonedaLemma |
| `RankNTypes` | forall внутри аргументов функций | Adjunctions, AlgebrasCoalgebras, BaseHaskell, ComonadTransformers, DistributedHaskell, FunctorHierarchy, GPUHaskell, KanExtensions, Monads, MonadTransformers, Optics, Profunctors, SubjectiveModeling, Toposes, TypeAlgebra, Uncertainty, YonedaLemma |
| `ExistentialQuantification` | скрытие типа за конструктором (forall в data) | KanExtensions, Profunctors, YonedaLemma |
| `FlexibleInstances` | инстансы для конкретных типов-применений | Adjunctions, Arrows, ComonadTransformers, FunctorHierarchy, KanExtensions, MetaProgramming, Optics, Profunctors, SubjectiveModeling, Toposes, Uncertainty, YonedaLemma |
| `FlexibleContexts` | произвольные ограничения в контекстах | Adjunctions, Arrows, ComonadTransformers, FunctorHierarchy, KanExtensions, MetaProgramming, Optics, Profunctors, SubjectiveModeling, Toposes, Uncertainty, YonedaLemma |
| `TypeSynonymInstances` | инстансы для синонимов типов | MetaProgramming |
| `MultiParamTypeClasses` | классы с несколькими параметрами | Adjunctions, ComonadTransformers, FunctorHierarchy, Profunctors, Toposes |
| `FunctionalDependencies` | зависимости параметров класса: \| m -> s | Adjunctions, FunctorHierarchy, Toposes |
| `InstanceSigs` | сигнатуры методов прямо в instance | Adjunctions, Arrows, FoldableTraversable, FunctorHierarchy, Optics, Profunctors, Toposes |
| `DefaultSignatures` | default-реализации с другой сигнатурой | MetaProgramming |
| `DeriveFunctor` | deriving Functor | Adjunctions, Comonads, ComonadTransformers, FoldableTraversable, Profunctors, SubjectiveModeling, Toposes, Uncertainty, YonedaLemma |
| `DeriveFoldable` | deriving Foldable | FoldableTraversable |
| `DeriveTraversable` | deriving Traversable | FoldableTraversable |
| `DeriveGeneric` | deriving Generic для GHC.Generics | MetaProgramming |
| `GeneralizedNewtypeDeriving` | newtype наследует инстансы обёрнутого типа | SubjectiveModeling, Toposes, Uncertainty |
| `TypeOperators` | операторы на уровне типов: f :+: g | Adjunctions, FunctorHierarchy, Optics, Profunctors |
| `TypeFamilies` | функции на уровне типов | Comonads, Toposes |
| `GADTs` | конструкторы с уточнёнными типами результата | Toposes |
| `DataKinds` | типы поднимаются в сорта (kinds) | Toposes |
| `PolyKinds` | полиморфизм по сортам | Toposes |
| `Arrows` | proc-нотация для стрелок | Arrows |
| `TemplateHaskell` | метапрограммирование: генерация кода при компиляции | MetaProgramming |
| `QuasiQuotes` | встраиваемые DSL-литералы [q\|...\|] | MetaProgramming |

---
**Следующий шаг:** [BaseHaskell.ipynb](BaseHaskell.ipynb)